In [2]:
import os
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from dotenv import load_dotenv

car_code = pd.read_excel("4054_차명코드(251223).xlsx")
car_code["cnmCode"]=car_code["코드"].astype(int).astype(str).str.zfill(6)


In [ ]:
# 아반떼 차명 모음 조회
df = pd.read_excel("4054_차명코드(251223).xlsx")
df["cnmCode"] = df["코드"].astype(int).astype(str).str.zfill(6)

keyword = "아반떼"
cand = df[df["코드명"].str.contains(keyword, na=False)][["cnmCode","코드명"]]
print(cand.sort_values("코드명").head(50))  # 후보 확인


In [5]:

def call_api(url, params):
    r = requests.get(url, params=params, timeout=30)
    txt = (r.text or "").strip()
    
    # XML 파싱
    try:
        root = ET.fromstring(txt)
    except ET.ParseError:
        return {"ok": False, "status": r.status_code, "text_head": txt[:300], "url": r.url}

    # 정상/에러 코드 읽기 (문서 예시 구조)  :contentReference[oaicite:4]{index=4}
    result_code = root.findtext("./header/resultCode")
    result_msg  = root.findtext("./header/resultMsg")
    dtaCo       = root.findtext("./body/dtaCo")

    return {"ok": True, "resultCode": result_code, "resultMsg": result_msg, "dtaCo": dtaCo, "url": r.url}

def crawling_car():
    load_dotenv()
    key = os.getenv("KEY")

    url = "https://apis.data.go.kr/B553881/newRegistlnfoService_02/getnewRegistlnfoService02"  # 문서 서비스 URL :contentReference[oaicite:5]{index=5}

    fuel_codes = ["2", "5", "7", "8"]
    sexes = ["남자", "여자"]
    months = [f"{m:02d}" for m in range(1, 3)]

    rows = []
    bad = []

    for agecode in range(1, 9):         # 10대~80대
        for mt in months:               # ✅ registMt 필수
            for fuel in fuel_codes:
                for sex in sexes:
                    params = {
                        "serviceKey": key,      
                        "registYy": "2025",     # 필수 :contentReference[oaicite:7]{index=7}
                        "registMt": mt,         # ✅ 필수 :contentReference[oaicite:8]{index=8}
                        "vhctyAsortCode": "1",
                        "registGrcCode": "1",
                        "useFuelCode": fuel,
                        "prposSeNm": "1",
                        "sexdstn": sex,
                        "agrde": str(agecode),
                        "prye": 2025,
                    }

                    res = call_api(url, params)

                    if not res["ok"] or res.get("resultCode") not in (None, "00"):
                        bad.append({
                            "agecode": agecode, "registMt": mt, "fuel": fuel, "sex": sex, "prye": 2025,
                            "status": res.get("status"), "resultCode": res.get("resultCode"),
                            "resultMsg": res.get("resultMsg"), "text_head": res.get("text_head"),
                        })
                        continue

                    rows.append({
                        "registYy": 2025, "registMt": mt, "agrde": agecode,
                        "useFuelCode": fuel, "sexdstn": sex, "prye": 2025,
                        "dtaCo": int(res["dtaCo"]) if res["dtaCo"] and res["dtaCo"].isdigit() else None

                    })

    df = pd.DataFrame(rows)
    bad_df = pd.DataFrame(bad)
    return df, bad_df

if __name__ == "__main__":
    df, bad_df = crawling_car()
    print(df.head())
    print("OK rows:", len(df))
    print("BAD rows:", len(bad_df))
    if len(bad_df):
        print(bad_df.head(10))


   registYy registMt  agrde useFuelCode sexdstn  prye  dtaCo
0      2025       01      1           2      남자  2025      4
1      2025       01      1           7      남자  2025      9
2      2025       01      1           7      여자  2025      3
3      2025       01      1           8      여자  2025      1
4      2025       02      1           2      남자  2025      2
OK rows: 108
BAD rows: 20
   agecode registMt fuel sex  prye status resultCode     resultMsg text_head
0        1       01    2  여자  2025   None         03  NODATA_ERROR      None
1        1       01    5  남자  2025   None         03  NODATA_ERROR      None
2        1       01    5  여자  2025   None         03  NODATA_ERROR      None
3        1       01    8  남자  2025   None         03  NODATA_ERROR      None
4        1       02    2  여자  2025   None         03  NODATA_ERROR      None
5        1       02    5  남자  2025   None         03  NODATA_ERROR      None
6        1       02    5  여자  2025   None         03  NODATA_ERROR   

In [31]:
df.to_csv("stocks.csv", index=False, encoding="cp949")

In [27]:
import os
import requests
from dotenv import load_dotenv

def test_one():
    load_dotenv()
    key = os.getenv("KEY")

    url = "https://apis.data.go.kr/B553881/newRegistlnfoService_02/getnewRegistlnfoService02"
    params = {
        "serviceKey": key,
        "registYy": "2025",
        "registMt": "01",
    }

    r = requests.get(url, params=params, timeout=30)
    print("status:", r.status_code)
    print("head:", (r.text or "")[:300])

if __name__ == "__main__":
    test_one()


status: 200
head: <response><header><resultCode>00</resultCode><resultMsg>NORMAL_CODE</resultMsg></header><body><dtaCo>124494</dtaCo></body></response>
